# Guided Target Analysis — LuminaPay

Build an activation-ready merchant population for an Instant Settlement campaign with a 6,000-contact capacity. The data, company, and monetary assumptions are synthetic.

**Decision boundary:** this workflow applies auditable business rules. It does not predict adoption or estimate causal lift.


## 1. Set up the reproducible workflow

Run this notebook from its own directory or the repository root. The helper locates the module and imports the reference functions.


In [ ]:
from pathlib import Path
import importlib.util
import pandas as pd

candidate_roots = [Path.cwd(), Path.cwd().parent, Path.cwd().parents[1]]
module_root = next(root / "03_target_analysis" for root in candidate_roots if (root / "03_target_analysis").exists())

def load_module(name, path):
    spec = importlib.util.spec_from_file_location(name, path)
    module = importlib.util.module_from_spec(spec)
    spec.loader.exec_module(module)
    return module

analysis = load_module("target_analysis", module_root / "src" / "analyze_targets.py")
generator = load_module("target_generator", module_root / "src" / "generate_synthetic_data.py")

full_path = module_root / "data" / "raw" / "lumina_settlement_target_full.csv"
raw = pd.read_csv(full_path) if full_path.exists() else generator.generate_raw_data()
raw.shape


## 2. Validate before targeting

Retain the raw report. Cleaning normalizes descriptive categories, labels missing industry, removes duplicate keys, and quarantines impossible numeric values.


In [ ]:
quality = analysis.quality_report(raw)
clean = analysis.clean_population(raw)
quality, clean.shape


## 3. Reconcile the eligibility funnel

Every rule is a hard gate. The sequential funnel explains volume movement; the final set is the intersection of all rules.


In [ ]:
funnel = analysis.eligibility_funnel(clean)
eligible = analysis.apply_eligibility(clean)
funnel.assign(share_of_total=lambda frame: frame["share_of_total"].map(lambda value: f"{value:.1%}"))


## 4. Score only eligible merchants

Need, value, and fit points are visible and reviewable. Country and industry receive no points.


In [ ]:
scored = analysis.score_population(eligible)
scored.groupby("priority_tier", observed=False).agg(
    eligible_merchants=("merchant_id", "size"),
    average_score=("priority_score", "mean"),
).sort_values("average_score", ascending=False)


## 5. Allocate the 6,000-contact capacity


In [ ]:
ranked = analysis.select_capacity(scored, capacity=6_000)
tier_mix = ranked.groupby("priority_tier", observed=False).agg(
    eligible=("merchant_id", "size"), selected=("selected", "sum")
)
opportunity = analysis.estimate_opportunity(ranked)
tier_mix, opportunity


## 6. Review operational coverage

Selection-rate differences are descriptive signals for review, not causal or fairness conclusions.


In [ ]:
country_review = analysis.segment_summary(ranked, "country")
industry_review = analysis.segment_summary(ranked, "industry")
country_review.round(3), industry_review.round(3)


## 7. Test the capacity tradeoff


In [ ]:
analysis.capacity_sensitivity(scored, capacities=(3_000, 6_000, 9_000)).round(3)


## 8. Activation and measurement handoff

Recommend the 6,000-merchant wave only after compliance and operations approve the versioned rules. Re-run recent-contact and enrollment suppressions immediately before send, export only required fields, and expire the list after seven days.

Within High and Medium priority bands, reserve a randomized holdout. Measure incremental adoption, not only observed response, alongside complaints, opt-outs, delivery failures, and segment coverage. The scenario value is an assumption until the experiment provides evidence.
